<a href="https://colab.research.google.com/github/DongAnYu/ai-tutor-rag-system/blob/main/notebooks/13-Adding_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [3]:
!pip install -qU llama-index==0.14.0 llama-index-llms-openai==0.5.6 openai==1.107.0 cohere==5.18.0 jedi==0.19.2 \
                 llama-index-llms-google-genai==0.5.0 chromadb==1.0.21 llama-index-vector-stores-chroma==0.5.3

In [4]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_OPENAI_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_GOOGLE_API_KEY>"

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')

In [5]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

nest_asyncio.apply()

# Load a Model


In [6]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


In [7]:
# Downloading Vector store from Hugging face hub
from huggingface_hub import hf_hub_download

vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [8]:
!unzip vectorstore.zip

Archive:  vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


In [9]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Create your index
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
vector_index = VectorStoreIndex.from_vector_store(vector_store)

In [10]:
# Query Engine
ai_tutor_knowledge_query_engine = vector_index.as_query_engine(similarity_top_k=3)

res = ai_tutor_knowledge_query_engine.query("How does Retrieval Augmented Generation (RAG) work?")
print(res.response)

Retrieval-Augmented Generation (RAG) is a hybrid approach that combines a retrieval system with a generative large language model to produce more accurate, up-to-date, and grounded responses. The workflow and components work together as follows:

- Two main components
  - Retrieval: extracts relevant information from external knowledge sources (documents, databases). It involves indexing (organizing documents for efficient search using sparse inverted indexes or dense vector encodings) and searching (fetching candidate documents for a query). Optional rerankers can refine the ordering of retrieved items.
  - Generation: a large language model uses the retrieved content plus the user query to produce the final response. Prompting techniques (e.g., Chain of Thought, Tree of Thought, rephrase-and-respond) guide the model to integrate retrieved evidence and reason without requiring model finetuning.

- Typical RAG processing steps
  1. Query classification: decide whether retrieval is nece

In [11]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("Metadata\t", src.metadata)
    print("-_" * 20)

Node ID	 2aa05360-f43a-4819-bce7-0acf7b897eab
Title	 Searching for Best Practices in Retrieval-Augmented Generation:1 Introduction
Text	 Generative large language models are prone to producing outdated information or fabricating facts, although they were aligned with human preferences by reinforcement learning [1] or lightweight alternatives [2–5]. Retrieval-augmented generation (RAG) techniques address these issues by combining the strengths of pretraining and retrieval-based models, thereby providing a robust framework for enhancing model performance [6]. Furthermore, RAG enables rapid deployment of applications for specific organizations and domains without necessitating updates to the model parameters, as long as query-related documents are provided. Many RAG approaches have been proposed to enhance large language models (LLMs) through query-dependent retrievals [6–8]. A typical RAG workflow usually contains multiple intervening processing steps: query classification (determining w

# Router

Routers are modules that take in a user query and a set of “choices” (defined by metadata), and returns one or more selected choices.

They can be used for the following use cases and more:

- Selecting the right data source among a diverse range of data sources

- Deciding whether to do summarization (e.g. using summary index query engine) or semantic search (e.g. using vector index query engine)

- Deciding whether to “try” out a bunch of choices at once and combine the results (using multi-routing capabilities).


## Lets create a different query engine with Mistral AI information


In [12]:
from pathlib import Path
import requests
import time

wiki_titles = [
    "Mistral AI",
    "Llama (language model)",
    "Claude AI",
    "OpenAI",
    "Gemini AI",
]

data_path = Path("llm_data_wiki")
if not data_path.exists():
    data_path.mkdir()

# Set up headers with User-Agent (REQUIRED by Wikipedia API)
headers = {
    'User-Agent': 'YourAppName/1.0 (your-email@example.com)'  # Replace with your info if this dummy gives an error
}

for title in wiki_titles:
    try:
        # Make the request with headers
        response = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "format": "json",
                "titles": title,
                "prop": "extracts",
                "explaintext": True,
            },
            headers=headers  # Add headers here
        )

        # Check if request was successful
        response.raise_for_status()

        if not response.text:
            print(f"Empty response for '{title}'")
            continue

        data = response.json()

        # Extract the page content
        if "query" in data and "pages" in data["query"]:
            page = next(iter(data["query"]["pages"].values()))
            if "extract" in page:
                wiki_text = page["extract"]
                with open(data_path / "llm_data_wiki.txt", "a", encoding="utf-8") as fp:
                    fp.write(f"Title: {title}\n{wiki_text}\n\n")
                print(f"Successfully saved: {title}")
            else:
                print(f"No extract found for '{title}'")
        else:
            print(f"Unexpected response format for '{title}'")
        time.sleep(0.5)

    except requests.exceptions.RequestException as e:
        print(f"Request error for '{title}': {e}")
    except ValueError as e:  # JSON decode error
        print(f"JSON decode error for '{title}': {e}")
        print(f"Response text: {response.text[:200]}...")

Successfully saved: Mistral AI
Successfully saved: Llama (language model)
Successfully saved: Claude AI
Successfully saved: OpenAI
Successfully saved: Gemini AI


In [13]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.text_splitter import TokenTextSplitter
from llama_index.core.extractors import (
    SummaryExtractor,
    QuestionsAnsweredExtractor,
    KeywordExtractor,
)

# Assuming you have prepared a directory for llm data
documents = SimpleDirectoryReader("llm_data_wiki").load_data()

text_splitter = TokenTextSplitter(separator=" ", chunk_size=512, chunk_overlap=128)

transformations = [
    text_splitter,
    QuestionsAnsweredExtractor(questions=2),
    SummaryExtractor(summaries=["prev", "self"]), # Summary of this chunk + previous chunk, Summary of this chunk alone
    KeywordExtractor(keywords=10),  # Extracts top 10 keywords per chunk
    OpenAIEmbedding(model="text-embedding-3-small"),
]

llm_index = VectorStoreIndex.from_documents(documents=documents, transformations=transformations)

llm_query_engine = llm_index.as_query_engine(similarity_top_k=2)

100%|██████████| 41/41 [00:16<00:00,  2.45it/s]


In [15]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import PydanticSingleSelector
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

# initialize tools
ai_tutor_knowledge_tool = QueryEngineTool.from_defaults(
    query_engine=ai_tutor_knowledge_query_engine,
    description="Useful for questions about general generative AI concepts",
)
llm_tool = QueryEngineTool.from_defaults(
    query_engine=llm_query_engine,
    description="Useful for questions about particular LLMs like Mistral, Claude, OpenAI, Gemini",
)

# initialize router query engine (single selection, pydantic)
query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),  # Selector — the routing brain
    query_engine_tools=[
        ai_tutor_knowledge_tool,
        llm_tool,
    ],
)

In [16]:
res = query_engine.query(
    "What is the LLama model?",
)
print(res.response)

The Llama series are autoregressive decoder‑only transformer large language models developed by Meta. They follow a GPT‑3–style architecture but use several different components: the SwiGLU activation function (instead of GeLU), rotary positional embeddings (RoPE) rather than absolute positional encodings, and RMSNorm instead of layer normalization. Meta emphasized scaling training-token volume (rather than just parameter count) when training Llama models. Llama foundational models were trained on very large public datasets (Llama 1 on ~1.4 trillion tokens from sources such as CommonCrawl, GitHub, multilingual Wikipedia, Project Gutenberg, Books3, arXiv, and Stack Exchange; Llama 2 on ~2 trillion tokens with more curation and upsampling of higher‑trust sources). Llama has been used as the basis for many downstream projects, tooling (e.g., the llama.cpp reimplementation), and applications (including deployments like “Space Llama” on the ISS).


In [17]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 d58559dc-459a-4a08-8f64-1fd600329422
Text	 which Meta denied.


=== Comparison of models ===
For the training cost column, only the largest model's cost is written by default. For example, "21,000" is the training cost of Llama 2 69B in units of petaFLOP-day. Also, 1 petaFLOP-day = 1 petaFLOP/sec × 1 day = 8.64E19 FLOP. "T" means "trillion" and "B" means "billion".
The following table lists the main model versions of Llama, describing the significant changes included with each version:


== Architecture and training ==


=== Architecture ===
Like GPT-3, the Llama series of models are autoregressive decoder-only transformers, but there are some minor differences:

Llama uses the SwiGLU activation function instead of GPT-3's GeLU.
Llama uses rotary positional embeddings (RoPE) instead of absolute positional embedding.
Instead of layer normalization, Llama uses RMSNorm.


=== Training datasets ===
Llama's developers focused their effort on scaling the model's performance by incre

In [18]:
res = query_engine.query("Explain parameter-efficient finetuning methods")
print(res.response)

Parameter-efficient fine-tuning (PEFT) refers to methods that adapt large pretrained models to new tasks while changing far fewer parameters than full fine-tuning. The goal is to save compute, memory, and storage (e.g., for many task-specific adapters) while preserving the pretrained model’s core knowledge. Three main approaches are commonly used:

1. Selective fine-tuning
- What it is: Only a subset of the model’s existing parameters are updated during training (e.g., a few layers or certain modules).
- Tradeoffs: Simple to implement and reduces training cost relative to full fine-tuning, but choosing which parameters to update is task- and model-dependent.

2. Reparameterization (example: LoRA)
- What it is: Replace or augment full-weight updates with a low-rank factorization of the weight change. Instead of learning a full Delta W, you learn two much smaller matrices A and B such that Delta W ≈ B A.
- How it reduces parameters: For a weight matrix W of size m x n, using rank r yield

In [19]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 6be88fa3-2f8b-43e7-aba0-d874b39809fc
Text	 # FourierFT: Discrete Fourier Transformation Fine-Tuning[FourierFT](https://huggingface.co/papers/2405.03003) is a parameter-efficient fine-tuning technique that leverages Discrete Fourier Transform to compress the model's tunable weights. This method outperforms LoRA in the GLUE benchmark and common ViT classification tasks using much less parameters.FourierFT currently has the following constraints:- Only `nn.Linear` layers are supported.- Quantized layers are not supported.If these constraints don't work for your use case, consider other methods instead.The abstract from the paper is:> Low-rank adaptation (LoRA) has recently gained much interest in fine-tuning foundation models. It effectively reduces the number of trainable parameters by incorporating low-rank matrices A and B to represent the weight change, i.e., Delta W=BA. Despite LoRA's progress, it faces storage challenges when handling extensive customization adaptations or 

# Function Agent using OpenAI GPT 5 Model


A function agent is an LLM that can orchestrate code execution as part of its reasoning loop.

In [20]:
system_message_openai_agent = """You are an AI teacher, answering questions from students of an applied AI course on Large Language Models (LLMs or llm) and Retrieval Augmented Generation (RAG) for LLMs. Topics covered include training models, fine-tuning models, giving memory to LLMs, prompting tips, hallucinations and bias, vector databases, transformer architectures, embeddings, RAG frameworks, Langchain, LlamaIndex, making LLMs interact with tools, AI agents, reinforcement learning with human feedback. Questions should be understood in this context.

Your answers are aimed to teach students, so they should be complete, clear, and easy to understand.

Use the available tools to gather insights pertinent to the field of AI. Always use two tools at the same time. These tools accept a string (a user query rewritten as a statement) and return informative content regarding the domain of AI.
e.g:
User question: 'How can I fine-tune an LLM?'
Input to the tool: 'Fine-tuning an LLM'

User question: How can quantize an LLM?
Input to the tool: 'Quantization for LLMs'

User question: 'Teach me how to build an AI agent"'
Input to the tool: 'Building an AI Agent'

Only some information returned by the tools might be relevant to the question, so ignore the irrelevant part and answer the question with what you have.

Your responses are exclusively based on the output provided by the tools. Refrain from incorporating information not directly obtained from the tool's responses.

When the conversation deepens or shifts focus within a topic, adapt your input to the tools to reflect these nuances. This means if a user requests further elaboration on a specific aspect of a previously discussed topic, you should reformulate your input to the tool to capture this new angle or more profound layer of inquiry.

Provide comprehensive answers, ideally structured in multiple paragraphs, drawing from the tool's variety of relevant details. The depth and breadth of your responses should align with the scope and specificity of the information retrieved.

Should the tools repository lack information on the queried topic, politely inform the user that the question transcends the bounds of your current knowledge base, citing the absence of relevant content in the tool's documentation.

At the end of your answers, always invite the students to ask deeper questions about the topic if they have any. Make sure to reformulate the question to the tool to capture this new angle or more profound layer of inquiry.

Do not refer to the documentation directly, but use the information provided within it to answer questions.

If code is provided in the information, share it with the students. It's important to provide complete code blocks so they can execute the code when they copy and paste them.

Make sure to format your answers in Markdown format, including code blocks and snippets.

Politely reject questions not related to AI, while being cautious not to reject unfamiliar terms or acronyms too quickly."""

In [22]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

# Initialize the LLM
llm = OpenAI(model="gpt-5", additional_kwargs={"reasoning_effort":"minimal"})

# Create the FunctionAgent
agent = FunctionAgent(
    tools=[ai_tutor_knowledge_tool, llm_tool],
    llm=llm,
    system_prompt=system_message_openai_agent,
    verbose=False  # no intermediate traces printed
)

# Run the agent queries
import asyncio

async def run_agent(query):
    response = await agent.run(query)
    return response

In [23]:
# Execute the async function
response = asyncio.run(run_agent("What is the LLama model?"))

print(response)

LLaMA (often written “Llama”) is a family of large language models developed by Meta AI. It’s designed as a set of foundation, decoder‑only Transformer models trained on large-scale public data and released in multiple parameter sizes so they can run on a range of hardware. The first LLaMA was announced on February 24, 2023, with a paper and blog post. It emphasized making capable models at different scales (e.g., 13B and 65B), with the 13B model reported to outperform much larger GPT‑3 baselines on many NLP benchmarks, and the 65B competitive with state-of-the-art at the time. While the initial inference code was released under GPLv3, access to weights was initially granted via application to researchers and institutions, and a third‑party torrent of the weights soon circulated widely. Subsequent releases (Llama 2, Llama 3, Llama 4) expanded the family; Llama 2 (July 18, 2023) arrived in 7B, 13B, and 70B sizes with an acceptable‑use license and included “Chat” variants aligned for ass

In [24]:
# Execute the async function
response = asyncio.run(run_agent("Explain parameter-efficient finetuning methods"))

print(response)

Parameter-efficient fine-tuning (PEFT) refers to techniques that adapt large pretrained language models using far fewer trainable parameters, less compute, and smaller storage than full-parameter fine-tuning. The goal is to enable customization (task adaptation, instruction tuning, domain adaptation, personalization) without re-training or storing a full copy of the model for each downstream task.

Common PEFT approaches and tradeoffs:

- Adapter modules
  - What: Small trainable feed-forward “bottleneck” layers inserted between transformer sublayers (e.g., after attention or MLP blocks). Only adapter weights are trained; the base model is frozen.
  - Pros: Very small extra parameters per task (often <<1% of the model), modular (can stack or compose), straightforward to implement.
  - Cons: Adds some inference cost from extra ops; capacity depends on adapter size.

- LoRA (Low-Rank Adaptation)
  - What: Replace updates to large weight matrices (e.g., attention and FFN projections) with

In [ ]:
# Execute the async function
response = asyncio.run(run_agent("Write the recipe for a chocolate cake."))

print(response)

I’m here to help with questions about AI, LLMs, and RAG. A chocolate cake recipe falls outside that scope, so I can’t provide it.

If you’d like, I can share how to build an AI assistant that retrieves recipes using RAG, or how to fine-tune an LLM on a corpus of cooking instructions to generate reliable recipes. For example:
- How to build a recipe RAG system (data ingestion, embeddings, vector search, prompt design)
- How to evaluate hallucinations in generated recipes
- How to make an agent that plans a cooking workflow and shopping list

Tell me which direction you prefer, and I’ll dive in. For instance, I can query: “Building a recipe RAG system with vector databases and LLMs” or “Fine-tuning an LLM on cooking instructions and preventing hallucinations.”


# RouterQueryEngine vs Function Agent (GPT-5) — Key Notes

## 1. Purpose

### RouterQueryEngine
- Routes queries between **multiple retrievers**
- Chooses **one query engine**
- Runs **one retrieval**
- Returns **one answer**

Use case:
- Multi-index RAG
- Domain-isolated QA
- Lightweight systems

---

### Function Agent (GPT-5)
- General **LLM-controlled agent**
- Plans actions
- Selects tools
- Can call **multiple tools**
- Chains steps and synthesizes results

Use case:
- Copilots
- Enterprise assistants
- Workflow automation
- Research agents

---

## 2. Execution Model

### RouterQueryEngine

User Query
↓
Selector
↓
Choose ONE engine
↓
Run retrieval once
↓
Answer

yaml
Copy code

Properties:
- Single step  
- Single tool  
- No loop  
- No planning  

---

### Function Agent

User Query
↓
GPT-5 planner
↓
Call tool A
↓
Result
↓
Call tool B (optional)
↓
Result
↓
Synthesize final answer

yaml
Copy code

Properties:
- Multi-step  
- Multi-tool  
- Iterative  
- Planning loop  

---

## 3. Routing Capability

### RouterQueryEngine
- Routes based on:
  - Query text  
  - Tool descriptions  
- Output:
  - Exactly **one tool**
  - Exactly **one call**

Limitations:
- No chaining  
- No combining sources  
- No retry / adaptation  

---

### Function Agent
- Routing inside full reasoning
- Can:
  - Use no tool  
  - Use one tool  
  - Use multiple tools  
  - Call tools sequentially  
  - Combine results  

Example:
- Call LLM tool → get architecture  
- Call AI tutor tool → get LoRA  
- Combine answers  

---

## 4. Tool Scope

### RouterQueryEngine
- Tools = **query engines only**
- Retrieval-only

Cannot handle:
- APIs  
- Databases  
- Calculators  
- Files  
- CAD  
- Evaluation  

---

### Function Agent
- Tools can be:
  - RAG engines  
  - SQL / DB tools  
  - Python functions  
  - REST APIs  
  - Calculators  
  - Evaluators  
  - CAD pipelines  

Enables:
- Automation  
- Workflows  
- Copilots  

---

## 5. Reasoning Power

### RouterQueryEngine
- Reasons only about:
  - Which tool to pick  
- No reasoning after retrieval  

---

### Function Agent
- Reasons about:
  - Intent  
  - Missing info  
  - Tool ordering  
  - Partial results  
  - Whether more tools are needed  
  - How to combine outputs  

Supports:
- Comparison  
- Multi-domain synthesis  
- Task planning  

---

## 6. Output Behavior

### RouterQueryEngine
- One source  
- One domain  
- Minimal synthesis  

---

### Function Agent
- Multi-source  
- Cross-domain  
- Fully synthesized by GPT-5  

---

## 7. Extensibility

### RouterQueryEngine
Strengths:
- Simple  
- Fast  
- Cheap  
- Deterministic  

Limits:
- Retrieval only  
- No automation  
- No chaining  

Best for:
- Multi-index QA  
- Teaching demos  
- Lightweight RAG  

---

### Function Agent
Strengths:
- General  
- Multi-tool  
- Multi-step  
- Automation-ready  

Tradeoffs:
- More complex  
- Slightly slower  
- Higher cost  

Best for:
- Copilots  
- Enterprise assistants  
- Research agents  
- Workflow automation  

---

## 8. Side-by-Side Summary

| Aspect | RouterQueryEngine | Function Agent |
|------|------------------|----------------|
| Purpose | Route to one retriever | General reasoning + orchestration |
| Steps | 1 | Many |
| Tool calls | Exactly 1 | 0, 1, or many |
| Planning | None | Full LLM planning |
| Chaining | ❌ | ✅ |
| Tool types | Query engines only | Any function / API / RAG |
| Reason over results | ❌ | ✅ |
| Automation | ❌ | ✅ |
| Agent loop | ❌ | ✅ |
| Copilot-grade | ❌ | ✅ |

---

## 9. Mapping to Your Systems

### Router system
- Multi-index RAG  
- Intent routing  
- Domain isolation  

Role:
- **Advanced retrieval router**

---

### Agent system
- GPT-5 planner  
- Tool-using agent  
- RAG as tools  
- Multi-step capable  

Role:
- **True agent architecture (copilot-grade)**

---

## 10. Final Takeaway

- RouterQueryEngine answers:  
  **“Which index should answer this question?”**

- Function Agent answers:  
  **“What actions should I take to solve this task?”**

Formal:

> RouterQueryEngine is a single-step retrieval router, while a Function Agent is a general LLM-controlled planning system that can select, chain, and orchestrate multiple tools to complete complex tasks.


# Code related questions to GPT-5, the remaining questions to Gemini

In [25]:
from llama_index.llms.openai import OpenAI
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import PydanticSingleSelector
from llama_index.core.tools import QueryEngineTool

# initialize LLMs
gpt_5_llm = OpenAI(model="gpt-5", additional_kwargs={"reasoning_effort":"minimal"})

gemini_llm = GoogleGenAI(model="gemini-2.5-flash", temperature=1, max_tokens=512)

# define query engines
llm_query_engine_code = vector_index.as_query_engine(
    llm=gpt_5_llm,
    similarity_top_k=3,
    embed_model=OpenAIEmbedding(model="text-embedding-3-small", mode="text_search"),
)

llm_query_engine_rest = vector_index.as_query_engine(
    llm=gemini_llm,
    similarity_top_k=3,
    embed_model=OpenAIEmbedding(model="text-embedding-3-small", mode="text_search"),
)

# define tools for LLM
llm_tool_code = QueryEngineTool.from_defaults(
    query_engine=llm_query_engine_code,
    description="Ideal for handling code-related queries, technical implementations, and troubleshooting involving Large Language Models.",
    name="LLMCodeTool",
)

llm_tool_rest = QueryEngineTool.from_defaults(
    query_engine=llm_query_engine_rest,
    description="Best suited for answering conceptual, theoretical, and general questions about Large Language Models.",
    name="LLMGeneralTool",

)


system_message_openai_agent_tools = """
You are a highly knowledgeable assistant specialized in Large Language Models. Your primary role is to assist users by providing accurate, detailed, and context-specific responses. You have access to two specialized tools:

1. **LLMCodeTool** – Use this tool when the query involves code-related tasks, technical implementations, debugging, or troubleshooting issues in code.
2. **LLMGeneralTool** – Use this tool for answering conceptual, theoretical, or general questions about Large Language Models that do not involve code specifics.

When a query is received:
- First, decide which tool best fits the user's request.
- If the question is technical or code-oriented, route the query to LLMCodeTool.
- If the question is more general or conceptual, route the query to LLMGeneralTool.
- If the query does not clearly fall into either category, provide a direct answer using your own capabilities.

Always ensure your responses are clear, concise, and directly address the user’s needs. Maintain a professional tone and provide detailed explanations where necessary.
"""
# Create the FunctionAgent
agent = FunctionAgent(
    tools=[llm_tool_code, llm_tool_rest],
    llm=gpt_5_llm,
    system_prompt=system_message_openai_agent_tools,
    verbose=False
)

# Run the agent queries
import asyncio

async def run_agent(query):
    response = await agent.run(query)

    return response.tool_calls[0].tool_name  # return which model is responding


In [26]:
# Execute the async function
response_code = asyncio.run(run_agent("How do I fine-tune the LLama model? Write the code for it"))

print(response_code)

LLMCodeTool


In [27]:
response_general = asyncio.run(run_agent("What is the relationship between Llama models and Meta"))

print(response_general)

LLMGeneralTool
